# FinClaw Local Fine-Tuning Pipeline
This notebook is tailored to run **locally on your Windows PC** with your **NVIDIA RTX 3050 (4 GB VRAM)** GPU.

### What this notebook does:
1. Verifies your local GPU and PyTorch CUDA setup.
2. Loads **Llama-3.2-1B-Instruct** in 4-bit QLoRA (caches on Drive D: to protect Drive C:).
3. Loads and formats **`finclaw_colab_dataset.jsonl`** directly from your local folder.
4. Trains locally in ~15–20 minutes using Hugging Face `SFTTrainer` + `bitsandbytes`.
5. Saves the fine-tuned adapter locally to `./finclaw_finetuned_adapter`.
6. Runs an immediate test generation in the final cell.


In [ ]:
# Step 1: Verify Local Hardware & PyTorch CUDA
import torch

print(f"PyTorch Version: {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Dedicated VRAM: {vram_gb:.2f} GB")
else:
    print("\n[!] NOTE: Please make sure you are running with the project's .venv kernel where CUDA PyTorch is installed!")


In [ ]:
# Step 2: Load Model & Tokenizer in 4-bit (QLoRA)
import os
import torch
# Ensure models download to D: drive instead of C: drive
os.environ['HF_HOME'] = os.path.abspath('.cache/huggingface')
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

MODEL_ID = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
print(f"Loading {MODEL_ID} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
    device_map="auto"
)

# Prepare model for PEFT LoRA training
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
print("Model loaded and prepared for QLoRA!")


In [ ]:
# Step 3: Load and Format Local Dataset
from datasets import load_dataset

DATASET_PATH = "finclaw_colab_dataset.jsonl"
print(f"Loading dataset from {DATASET_PATH} ...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

def formatting_prompts_func(examples):
    convs = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convs]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Total training examples: {len(dataset)}")
print("Sample conversation preview:")
print(dataset[0]["text"][:350])


In [ ]:
# Step 4: Train Locally with SFTTrainer
from trl import SFTTrainer, SFTConfig

output_dir = "./finclaw_local_checkpoints"

sft_config = SFTConfig(
    output_dir=output_dir,
    dataset_text_field="text",
    max_length=1024,
    per_device_train_batch_size=1,       # 1 ensures safety on 4GB VRAM
    gradient_accumulation_steps=4,      # effective batch size = 4
    warmup_steps=15,
    max_steps=350,                      # 350 steps (~2-3 epochs over 758 examples)
    learning_rate=2e-4,
    bf16=True,
    fp16=False,
    logging_steps=25,
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=100,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config
)

print("Starting local training on your PC...")
trainer.train()
print("Training finished successfully!")


In [ ]:
# Step 5: Save Model Locally
save_path = "./finclaw_finetuned_adapter"
print(f"Saving trained model adapter to {save_path} ...")
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Saved! You can find your fine-tuned model files right here in: {save_path}")


In [ ]:
# Step 6: Test Inference Locally with the Fine-Tuned Model
prompt = "Explain the financial implications of increasing working capital for a manufacturing firm."

messages = [
    {"role": "system", "content": "You are FinClaw, an expert AI financial analyst."},
    {"role": "user", "content": prompt}
]

formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_input, return_tensors="pt").to("cuda")

print("Prompt:", prompt)
print("Generating response...")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("\n--- FinClaw Response ---")
print(response)
